# TensorFlow & Keras: From Basics to Advanced

**Purpose:** Build fluency in TensorFlow so you can read and understand deep learning code in ML/AI projects.

TensorFlow (with its high-level API **Keras**) is one of the two dominant deep learning frameworks (alongside PyTorch). TensorFlow dominates production/deployment, while PyTorch dominates research — but you'll encounter both.

**How this notebook is organized:**
1. Tensors — Creating, Properties, and dtypes
2. Tensor Operations — Math, Indexing, and Reshaping
3. NumPy ↔ TensorFlow Interop
4. GradientTape — Automatic Differentiation
5. Keras Layers — The Building Blocks
6. Building Models (Sequential, Functional, Subclassing)
7. Compiling and Training (`model.compile`, `model.fit`)
8. Callbacks (EarlyStopping, Checkpoints, LR Scheduling)
9. Data Pipelines (`tf.data.Dataset`)
10. Saving and Loading Models
11. GPU / Device Management
12. Common Architectures (CNN, RNN, Transformer patterns)
13. Real Patterns You'll See in AI Code

In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TF info/warning messages
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'  # Suppress oneDNN messages

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version:      {keras.__version__}")
print(f"GPU available:      {len(tf.config.list_physical_devices('GPU')) > 0}")

TensorFlow version: 2.21.0
Keras version:      3.13.2


GPU available:      False


## 1. Tensors — Creating, Properties, and dtypes

TensorFlow tensors are similar to NumPy arrays but are **immutable** (you can't change individual elements) and can run on GPUs.

```
NumPy:       np.array     →  ndarray     (mutable)
PyTorch:     torch.tensor →  Tensor      (mutable)
TensorFlow:  tf.constant  →  EagerTensor (immutable)
             tf.Variable  →  Variable    (mutable — used for model weights)
```

In [2]:
# tf.constant — the basic tensor (immutable)
a = tf.constant([1, 2, 3, 4, 5])
print("1D tensor:", a)

# 2D tensor (matrix)
m = tf.constant([[1, 2, 3],
                 [4, 5, 6]])
print("\n2D tensor:\n", m)

# With explicit dtype
floats = tf.constant([1, 2, 3], dtype=tf.float32)
print("\nFloat32:", floats)

# tf.Variable — mutable tensor (for model weights)
v = tf.Variable([1.0, 2.0, 3.0])
print("\nVariable:", v)
v.assign([10.0, 20.0, 30.0])  # Can change values
print("After assign:", v)

1D tensor: tf.Tensor([1 2 3 4 5], shape=(5,), dtype=int32)

2D tensor:
 tf.Tensor(
[[1 2 3]
 [4 5 6]], shape=(2, 3), dtype=int32)

Float32: tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32)

Variable: <tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([1., 2., 3.], dtype=float32)>
After assign: <tf.Variable 'Variable:0' shape=(3,) dtype=float32, numpy=array([10., 20., 30.], dtype=float32)>


In [3]:
# Factory functions — same concepts as NumPy/PyTorch
zeros = tf.zeros((3, 4))            # np.zeros((3, 4))
ones = tf.ones((2, 3))              # np.ones((2, 3))
full = tf.fill((2, 2), 7.0)         # np.full((2, 2), 7.0)
eye = tf.eye(3)                      # np.eye(3)
rng = tf.range(0, 10, 2)            # np.arange(0, 10, 2)
lin = tf.linspace(0.0, 1.0, 5)      # np.linspace(0, 1, 5)

print("zeros:\n", zeros)
print("range:", rng)
print("linspace:", lin)

# Random tensors
rand_uniform = tf.random.uniform((3, 4))               # Uniform [0, 1)
rand_normal = tf.random.normal((3, 4))                  # Normal N(0,1)
rand_int = tf.random.uniform((3, 4), 0, 10, dtype=tf.int32)  # Random ints

print("\nrandom normal (3x4):\n", rand_normal)

zeros:
 tf.Tensor(
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]], shape=(3, 4), dtype=float32)
range: tf.Tensor([0 2 4 6 8], shape=(5,), dtype=int32)
linspace: tf.Tensor([0.   0.25 0.5  0.75 1.  ], shape=(5,), dtype=float32)

random normal (3x4):
 tf.Tensor(
[[-0.45418796  0.3831932  -0.46302363 -0.3829128 ]
 [-0.31570655  0.5563008   0.7136187   1.2966149 ]
 [-1.5643067   0.5397248   1.1015408   0.08441747]], shape=(3, 4), dtype=float32)


In [4]:
# Tensor properties
t = tf.random.normal((3, 4))

print("shape:", t.shape)        # TensorShape([3, 4])
print("dtype:", t.dtype)        # tf.float32 (default, same as PyTorch)
print("ndim: ", t.ndim)         # 2
print("size: ", tf.size(t).numpy())  # 12 — total elements

# Common dtypes:
# tf.float32 (default) — model weights, features
# tf.float16           — mixed precision training
# tf.int32, tf.int64   — indices, labels
# tf.bool              — masks
# tf.string            — text data in tf.data pipelines

# Type casting — tf.cast() instead of .astype()
a = tf.constant([1, 2, 3])
print("\nOriginal:", a.dtype)
print("To float:", tf.cast(a, tf.float32).dtype)
print("To bool: ", tf.cast(a, tf.bool))

shape: (3, 4)
dtype: <dtype: 'float32'>
ndim:  2
size:  12

Original: <dtype: 'int32'>
To float: <dtype: 'float32'>
To bool:  tf.Tensor([ True  True  True], shape=(3,), dtype=bool)


## 2. Tensor Operations — Math, Indexing, and Reshaping

In [5]:
# Element-wise math — same operators as NumPy/PyTorch
a = tf.constant([1.0, 2.0, 3.0, 4.0])
b = tf.constant([10.0, 20.0, 30.0, 40.0])

print("a + b:  ", (a + b).numpy())
print("a * b:  ", (a * b).numpy())       # Element-wise
print("a ** 2: ", (a ** 2).numpy())
print("a + 10: ", (a + 10).numpy())      # Scalar broadcast

# Common math functions
print("\nsqrt:  ", tf.sqrt(a).numpy())
print("exp:   ", tf.exp(a).numpy())
print("log:   ", tf.math.log(a).numpy())
print("clip:  ", tf.clip_by_value(a, 1.5, 3.5).numpy())  # Like np.clip

a + b:   [11. 22. 33. 44.]
a * b:   [ 10.  40.  90. 160.]
a ** 2:  [ 1.  4.  9. 16.]
a + 10:  [11. 12. 13. 14.]

sqrt:   [1.        1.4142135 1.7320508 2.       ]
exp:    [ 2.7182817  7.389056  20.085537  54.59815  ]
log:    [0.        0.6931472 1.0986123 1.3862944]
clip:   [1.5 2.  3.  3.5]


In [6]:
# Aggregation — uses axis= (same as NumPy, NOT dim= like PyTorch)
t = tf.constant([[1.0, 2.0, 3.0],
                 [4.0, 5.0, 6.0]])

print("sum:     ", tf.reduce_sum(t).numpy())
print("mean:    ", tf.reduce_mean(t).numpy())
print("max:     ", tf.reduce_max(t).numpy())
print("argmax:  ", tf.argmax(t, axis=1).numpy())  # Per-row max index

# axis=0: collapse rows (down columns)
# axis=1: collapse columns (across rows)
print("\nsum axis=0:", tf.reduce_sum(t, axis=0).numpy())   # [5, 7, 9]
print("sum axis=1:", tf.reduce_sum(t, axis=1).numpy())     # [6, 15]
print("mean axis=0:", tf.reduce_mean(t, axis=0).numpy())

# TF naming: tf.reduce_sum, tf.reduce_mean, tf.reduce_max
# The "reduce_" prefix is TF's convention — same operation as NumPy/PyTorch

sum:      21.0
mean:     3.5
max:      6.0
argmax:   [2 2]

sum axis=0: [5. 7. 9.]
sum axis=1: [ 6. 15.]
mean axis=0: [2.5 3.5 4.5]


In [7]:
# Indexing and slicing — same as NumPy
m = tf.constant([[1,  2,  3,  4],
                 [5,  6,  7,  8],
                 [9, 10, 11, 12]])

print("Element [1,2]:", m[1, 2].numpy())        # 7
print("Row 0:        ", m[0].numpy())            # [1, 2, 3, 4]
print("Column 1:     ", m[:, 1].numpy())         # [2, 6, 10]
print("Submatrix:\n",   m[0:2, 1:3].numpy())    # rows 0-1, cols 1-2

# Boolean indexing — use tf.boolean_mask
a = tf.constant([10, 25, 5, 30, 15])
mask = a > 12
print("\nMask:", mask.numpy())
print("Filtered:", tf.boolean_mask(a, mask).numpy())  # [25, 30, 15]

# tf.where — conditional selection (like np.where)
result = tf.where(a > 12, a, tf.zeros_like(a))
print("where:  ", result.numpy())

Element [1,2]: 7
Row 0:         [1 2 3 4]
Column 1:      [ 2  6 10]
Submatrix:
 [[2 3]
 [6 7]]

Mask: [False  True False  True  True]
Filtered: [25 30 15]
where:   [ 0 25  0 30 15]


In [8]:
# Reshaping
a = tf.range(12)

# tf.reshape — same as NumPy, -1 means "infer"
print("reshape(3,4):\n", tf.reshape(a, (3, 4)).numpy())
print("reshape(-1,3):", tf.reshape(a, (-1, 3)).shape)

# Expand/squeeze dims — TF equivalents of PyTorch unsqueeze/squeeze
t = tf.constant([1, 2, 3])                    # shape: (3,)
print("\nOriginal shape:", t.shape)
print("expand_dims(0):", tf.expand_dims(t, 0).shape)   # (1, 3) — add batch dim
print("expand_dims(1):", tf.expand_dims(t, 1).shape)   # (3, 1)

t2 = tf.random.normal((1, 3, 1))
print("squeeze:       ", tf.squeeze(t2).shape)          # (3,)

# Transpose
m = tf.random.normal((2, 3))
print("\nTranspose:", tf.transpose(m).shape)      # (3, 2)

# Permute dimensions (for images: NHWC ↔ NCHW)
img = tf.random.normal((8, 32, 32, 3))   # TF default: (batch, H, W, C)
print("Original:", img.shape)
print("Permuted:", tf.transpose(img, perm=[0, 3, 1, 2]).shape)  # → (8, 3, 32, 32)

# Concatenate
a = tf.random.normal((2, 3))
b = tf.random.normal((2, 3))
print("\nconcat axis=0:", tf.concat([a, b], axis=0).shape)  # (4, 3)
print("concat axis=1:", tf.concat([a, b], axis=1).shape)    # (2, 6)
print("stack:        ", tf.stack([a, b], axis=0).shape)      # (2, 2, 3)

reshape(3,4):
 [[ 0  1  2  3]
 [ 4  5  6  7]
 [ 8  9 10 11]]
reshape(-1,3): (4, 3)

Original shape: (3,)
expand_dims(0): (1, 3)
expand_dims(1): (3, 1)
squeeze:        (3,)

Transpose: (3, 2)
Original: (8, 32, 32, 3)
Permuted: (8, 3, 32, 32)

concat axis=0: (4, 3)
concat axis=1: (2, 6)
stack:         (2, 2, 3)


In [9]:
# Matrix multiplication
a = tf.random.normal((3, 4))
b = tf.random.normal((4, 5))

c1 = a @ b                    # Operator (most common)
c2 = tf.matmul(a, b)          # Function

print("a @ b shape:", c1.shape)     # (3, 5)
print("All equal:  ", tf.reduce_all(tf.equal(c1, c2)).numpy())

# Batch matmul — same as PyTorch
batch_a = tf.random.normal((8, 3, 4))
batch_b = tf.random.normal((8, 4, 5))
result = batch_a @ batch_b
print("\nBatch matmul:", result.shape)  # (8, 3, 5)

a @ b shape: (3, 5)
All equal:   True

Batch matmul: (8, 3, 5)


## 3. NumPy ↔ TensorFlow Interop

Seamless conversion — TensorFlow was designed to work closely with NumPy.

In [10]:
# NumPy → TensorFlow
np_array = np.array([1.0, 2.0, 3.0])
tensor = tf.constant(np_array)            # Creates a copy
tensor2 = tf.convert_to_tensor(np_array)  # Same thing, more explicit
print("From NumPy:", tensor)

# TensorFlow → NumPy
t = tf.constant([4.0, 5.0, 6.0])
np_back = t.numpy()                       # .numpy() extracts the value
print("To NumPy:  ", np_back, type(np_back))

# TF tensors work directly with NumPy functions (eager mode)
result = np.multiply(t, 2)
print("np.multiply:", result)

# Common pattern: pandas → numpy → tensor
# X_tensor = tf.constant(df[feature_cols].values, dtype=tf.float32)
# y_tensor = tf.constant(df['target'].values, dtype=tf.int32)

# Extract a scalar value
scalar = tf.reduce_sum(t)
print("\nScalar tensor:", scalar)
print("Python float: ", scalar.numpy())  # Like PyTorch's .item()

From NumPy: tf.Tensor([1. 2. 3.], shape=(3,), dtype=float64)
To NumPy:   [4. 5. 6.] <class 'numpy.ndarray'>
np.multiply: [ 8. 10. 12.]

Scalar tensor: tf.Tensor(15.0, shape=(), dtype=float32)
Python float:  15.0


## 4. GradientTape — Automatic Differentiation

TensorFlow's equivalent of PyTorch's autograd. Instead of `requires_grad=True` + `.backward()`, TF uses a `GradientTape` context manager.

In [11]:
# GradientTape records operations for automatic differentiation
x = tf.Variable([2.0, 3.0])  # Must be a Variable to compute gradients

with tf.GradientTape() as tape:
    y = x ** 2 + 3 * x    # y = x² + 3x
    loss = tf.reduce_sum(y)

# Compute gradients: dy/dx = 2x + 3
gradients = tape.gradient(loss, x)
print("x:        ", x.numpy())
print("y:        ", y.numpy())
print("Gradients:", gradients.numpy())  # [7.0, 9.0] ✓

# KEY DIFFERENCES from PyTorch:
# PyTorch:    loss.backward()     → gradients stored in x.grad
# TensorFlow: tape.gradient(loss, x) → gradients returned as output
# 
# PyTorch:    requires_grad=True on tensors
# TensorFlow: must use tf.Variable (tf.constant won't track gradients)
#
# The tape is consumed after one .gradient() call (one-time use by default)

x:         [2. 3.]
y:         [10. 18.]
Gradients: [7. 9.]


In [12]:
# Custom training loop with GradientTape
# You'll see this pattern in advanced TF code (research, custom training)

# Simple linear model: y = Wx + b
W = tf.Variable(tf.random.normal((1,)))
b = tf.Variable(tf.zeros((1,)))

x_data = tf.constant([1.0, 2.0, 3.0, 4.0])
y_true = tf.constant([3.0, 5.0, 7.0, 9.0])  # y = 2x + 1
lr = 0.01

for step in range(200):
    with tf.GradientTape() as tape:
        y_pred = W * x_data + b
        loss = tf.reduce_mean((y_pred - y_true) ** 2)
    
    # Compute gradients for BOTH W and b
    gradients = tape.gradient(loss, [W, b])
    
    # Manual gradient descent update
    W.assign_sub(lr * gradients[0])  # W = W - lr * dL/dW
    b.assign_sub(lr * gradients[1])  # b = b - lr * dL/db

print(f"Learned: y = {W.numpy()[0]:.2f}x + {b.numpy()[0]:.2f}")  # Should be ≈ 2x + 1
print(f"Final loss: {loss.numpy():.6f}")

Learned: y = 2.10x + 0.72
Final loss: 0.013609


## 5. Keras Layers — The Building Blocks

Keras layers are TensorFlow's equivalent of PyTorch's `nn.Module` layers. They're the LEGO bricks of neural networks.

In [13]:
# Dense (fully connected) layer — equivalent of PyTorch's nn.Linear
dense = layers.Dense(units=5, activation='relu')
x = tf.random.normal((32, 10))    # batch of 32, 10 features
out = dense(x)
print("Dense: input", x.shape, "→ output", out.shape)  # (32, 5)

# Weights are created on first call (lazy initialization)
print("Weight shape:", dense.kernel.shape)    # (10, 5) — TF calls it 'kernel'
print("Bias shape:  ", dense.bias.shape)      # (5,)

# Activation is BUILT INTO the layer (different from PyTorch!)
# layers.Dense(64, activation='relu')  ← TF style (common)
# layers.Dense(64) followed by layers.Activation('relu')  ← also valid

# Common activations:
# 'relu'     — default for hidden layers
# 'sigmoid'  — binary classification output
# 'softmax'  — multi-class classification output
# 'tanh'     — sometimes in RNNs
# None       — no activation (linear, used for output layers before loss)

Dense: input (32, 10) → output (32, 5)
Weight shape: (10, 5)
Bias shape:   (5,)


In [14]:
# Other common layers you'll encounter

# Dropout — randomly zeros elements during training
dropout = layers.Dropout(rate=0.5)  # Note: 'rate' (TF) vs 'p' (PyTorch) — same meaning
x = tf.ones((1, 10))
print("Dropout (training):", dropout(x, training=True).numpy())
print("Dropout (eval):    ", dropout(x, training=False).numpy())  # All pass through

# BatchNormalization
bn = layers.BatchNormalization()
x = tf.random.normal((32, 5))
print("\nBatchNorm:", bn(x, training=True).shape)

# Embedding — integer IDs to dense vectors
embed = layers.Embedding(input_dim=1000, output_dim=64)
word_ids = tf.constant([5, 42, 999, 0])
print("Embedding:", embed(word_ids).shape)  # (4, 64)

# Flatten — collapse spatial dimensions
flatten = layers.Flatten()
x = tf.random.normal((8, 7, 7, 32))  # batch of feature maps
print("\nFlatten:", flatten(x).shape)  # (8, 1568)

# KEY DIFFERENCE from PyTorch:
# TF layers take training=True/False as argument
# PyTorch uses model.train() / model.eval() to switch modes globally

Dropout (training): [[2. 2. 0. 2. 2. 0. 0. 0. 2. 0.]]
Dropout (eval):     [[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]]

BatchNorm: (32, 5)
Embedding: (4, 64)

Flatten: (8, 1568)


## 6. Building Models — Three Ways

Keras offers three ways to build models. You'll encounter ALL three in the wild.

| Style | When to use | Flexibility |
|---|---|---|
| **Sequential** | Simple stack of layers | Low (no branching) |
| **Functional** | Multi-input/output, branching, sharing | Medium (most common) |
| **Subclassing** | Full Python control (like PyTorch) | High (research code) |

In [15]:
# STYLE 1: Sequential — simplest, for linear stacks
# Equivalent to PyTorch's nn.Sequential

model_seq = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(3, activation='softmax')  # 3 classes
])

model_seq.summary()  # Shows architecture, shapes, and parameter counts

C:\Users\Windows11\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_1 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,699 (22.26 KB)

 Trainable params: 5,699 (22.26 KB)

 Non-trainable params: 0 (0.00 B)

In [16]:
# STYLE 2: Functional API — the most common in production code
# Allows multiple inputs/outputs, shared layers, skip connections

inputs = keras.Input(shape=(20,))         # Define input shape
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dropout(0.3)(x)
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(3, activation='softmax')(x)

model_func = keras.Model(inputs=inputs, outputs=outputs, name="functional_model")
model_func.summary()

Model: "functional_model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 20)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,699 (22.26 KB)

 Trainable params: 5,699 (22.26 KB)

 Non-trainable params: 0 (0.00 B)

In [17]:
# STYLE 3: Model subclassing — most like PyTorch
# Full Python control, used in research and custom architectures

class SimpleNet(keras.Model):
    def __init__(self, hidden_dim, num_classes):
        super().__init__()
        self.dense1 = layers.Dense(hidden_dim, activation='relu')
        self.dense2 = layers.Dense(hidden_dim, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.out = layers.Dense(num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        # 'call' in TF = 'forward' in PyTorch
        x = self.dense1(inputs)
        x = self.dropout(x, training=training)  # Must pass training flag
        x = self.dense2(x)
        return self.out(x)

model_sub = SimpleNet(hidden_dim=64, num_classes=3)

# Build the model by passing dummy data (needed for summary)
model_sub(tf.random.normal((1, 20)))
model_sub.summary()

# Comparison:
# PyTorch:     class Model(nn.Module)  → def forward(self, x)
# TensorFlow:  class Model(keras.Model) → def call(self, inputs, training=False)

Model: "simple_net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_7 (Dense)                 │ (1, 64)                │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (1, 64)                │         4,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (1, 3)                 │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,699 (22.26 KB)

 Trainable params: 5,699 (22.26 KB)

 Non-trainable params: 0 (0.00 B)

## 7. Compiling and Training — `model.compile` + `model.fit`

**This is the killer feature of Keras.** The entire training loop in 2 lines. PyTorch requires you to write the loop manually; Keras handles it for you.

In [18]:
# Step 1: Build model
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu'),
    layers.Dense(3, activation='softmax')
])

# Step 2: Compile — specify loss, optimizer, and metrics
model.compile(
    optimizer='adam',                           # or keras.optimizers.Adam(learning_rate=0.001)
    loss='sparse_categorical_crossentropy',    # Integer labels (0, 1, 2, ...)
    metrics=['accuracy']                        # Track during training
)

# Common loss functions:
# 'sparse_categorical_crossentropy' — multi-class, integer labels (most common)
# 'categorical_crossentropy'        — multi-class, one-hot labels
# 'binary_crossentropy'             — binary classification
# 'mse'                             — regression (mean squared error)

# Common optimizers:
# 'adam'       — default choice
# 'sgd'       — simpler, sometimes better
# keras.optimizers.Adam(learning_rate=0.001)  — with custom LR

print("Model compiled successfully")

Model compiled successfully


In [19]:
# Step 3: Train with model.fit() — ONE LINE does the entire training loop!
# (Forward pass, loss, backward, optimizer step, metrics — all automatic)

# Generate fake data
np.random.seed(42)
X_train = np.random.randn(500, 20).astype(np.float32)
y_train = np.random.randint(0, 3, 500)
X_val = np.random.randn(100, 20).astype(np.float32)
y_val = np.random.randint(0, 3, 100)

# Train!
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    validation_data=(X_val, y_val),  # Evaluate on validation set each epoch
    verbose=1                         # 1=progress bar, 0=silent, 2=one line per epoch
)

# history.history contains all metrics per epoch
print("\nHistory keys:", list(history.history.keys()))

Epoch 1/10


 1/16 ━━━━━━━━━━━━━━━━━━━━ 6s 450ms/step - accuracy: 0.4375 - loss: 1.1202

16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.3280 - loss: 1.1506 - val_accuracy: 0.4100 - val_loss: 1.1056


Epoch 2/10


 1/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3750 - loss: 1.1365

...

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4840 - loss: 1.0339 - val_accuracy: 0.3300 - val_loss: 1.1461


Epoch 10/10


 1/16 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.5000 - loss: 1.0229

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4780 - loss: 1.0310 - val_accuracy: 0.3400 - val_loss: 1.1504



History keys: ['accuracy', 'loss', 'val_accuracy', 'val_loss']


In [20]:
# Step 4: Evaluate and predict

# Evaluate on test data
X_test = np.random.randn(100, 20).astype(np.float32)
y_test = np.random.randint(0, 3, 100)

test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc:  {test_acc:.2%}")

# Predict — returns probabilities (because we used softmax)
predictions = model.predict(X_test[:5], verbose=0)
print(f"\nPrediction shape: {predictions.shape}")   # (5, 3) — 5 samples, 3 class probabilities
print(f"First prediction: {predictions[0]}")
print(f"Predicted class:  {np.argmax(predictions[0])}")

# Get class labels for all predictions
predicted_classes = np.argmax(predictions, axis=1)
print(f"All predicted:    {predicted_classes}")

Test Loss: 1.0752
Test Acc:  43.00%



Prediction shape: (5, 3)
First prediction: [0.253233   0.19323874 0.55352825]
Predicted class:  2
All predicted:    [2 1 1 0 0]


## 8. Callbacks — Hooks into the Training Loop

Callbacks let you add behavior during training without modifying the training loop. You'll see these in every serious Keras project.

In [21]:
# The 3 most common callbacks:

# 1. EarlyStopping — stop when validation loss stops improving
early_stop = keras.callbacks.EarlyStopping(
    monitor='val_loss',     # What metric to watch
    patience=5,             # How many epochs to wait
    restore_best_weights=True  # Roll back to the best epoch
)

# 2. ModelCheckpoint — save the best model during training
# checkpoint = keras.callbacks.ModelCheckpoint(
#     'best_model.keras',
#     monitor='val_loss',
#     save_best_only=True
# )

# 3. ReduceLROnPlateau — reduce learning rate when metric plateaus
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,        # Multiply LR by this factor
    patience=3,        # Wait this many epochs before reducing
    min_lr=1e-6
)

# TensorBoard — log metrics for visualization
# tb = keras.callbacks.TensorBoard(log_dir='./logs')

# Use callbacks in model.fit()
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop, reduce_lr],  # Pass as a list
    verbose=0
)

print(f"Stopped at epoch {len(history.history['loss'])} (early stopping with patience=5)")
print(f"Best val_loss: {min(history.history['val_loss']):.4f}")

Stopped at epoch 8 (early stopping with patience=5)
Best val_loss: 1.1732


## 9. Data Pipelines — `tf.data.Dataset`

TensorFlow's data loading system. More powerful than PyTorch's DataLoader for complex pipelines, but more verbose for simple cases.

In [22]:
# Create dataset from numpy arrays (most common starting point)
X = np.random.randn(200, 10).astype(np.float32)
y = np.random.randint(0, 3, 200)

dataset = tf.data.Dataset.from_tensor_slices((X, y))
print("Dataset:", dataset)
print("Element spec:", dataset.element_spec)

# Chain transformations — the functional pipeline pattern
train_ds = (
    dataset
    .shuffle(buffer_size=1000)   # Shuffle with a buffer
    .batch(32)                    # Batch into groups of 32
    .prefetch(tf.data.AUTOTUNE)  # Prefetch next batch while training
)

# Iterate over batches
for batch_x, batch_y in train_ds.take(2):  # .take(n) = first n batches
    print(f"Batch: features={batch_x.shape}, labels={batch_y.shape}")

Dataset: <_TensorSliceDataset element_spec=(TensorSpec(shape=(10,), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))>
Element spec: (TensorSpec(shape=(10,), dtype=tf.float32, name=None), TensorSpec(shape=(), dtype=tf.int32, name=None))
Batch: features=(32, 10), labels=(32,)
Batch: features=(32, 10), labels=(32,)


In [23]:
# .map() — apply transformations to each element (like data augmentation)
def normalize(features, label):
    features = (features - tf.reduce_mean(features)) / tf.math.reduce_std(features)
    return features, label

train_ds_normalized = (
    dataset
    .map(normalize, num_parallel_calls=tf.data.AUTOTUNE)  # Parallel processing
    .shuffle(1000)
    .batch(32)
    .prefetch(tf.data.AUTOTUNE)
)

# Use tf.data.Dataset directly with model.fit()
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(10,)),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

history = model.fit(train_ds_normalized, epochs=3, verbose=1)

# The standard tf.data pipeline:
# 1. from_tensor_slices() or from_generator() — create dataset
# 2. .map()     — preprocessing/augmentation
# 3. .shuffle() — randomize order
# 4. .batch()   — group into batches
# 5. .prefetch() — overlap data loading with training

Epoch 1/3


1/7 ━━━━━━━━━━━━━━━━━━━━ 1s 304ms/step - accuracy: 0.2500 - loss: 1.2752

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3100 - loss: 1.2430  


Epoch 2/3


1/7 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4375 - loss: 1.1576

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3150 - loss: 1.1945 


Epoch 3/3


1/7 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.3438 - loss: 1.2640

7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - accuracy: 0.3550 - loss: 1.1652 


## 10. Saving and Loading Models

In [24]:
# TF/Keras has the SIMPLEST save/load of any framework

# Option 1: Save entire model (architecture + weights + optimizer state)
# model.save('my_model.keras')           # New Keras 3 format (recommended)
# model.save('my_model.h5')              # Legacy HDF5 format (still common)
# loaded_model = keras.models.load_model('my_model.keras')

# Option 2: Save weights only (must rebuild architecture first)
# model.save_weights('weights.weights.h5')
# model.load_weights('weights.weights.h5')

# Option 3: SavedModel format (for TF Serving / production deployment)
# model.export('saved_model_dir')
# loaded = tf.saved_model.load('saved_model_dir')

# Inspect model weights
print("Model layers and shapes:")
for layer in model.layers:
    weights = layer.get_weights()
    if weights:
        print(f"  {layer.name}: {[w.shape for w in weights]}")

print("\n(Save/load commands commented out to avoid file creation)")

# Comparison:
# PyTorch:     torch.save(model.state_dict(), 'model.pth')  — manual
# TensorFlow:  model.save('model.keras')                    — one line, saves everything

Model layers and shapes:
  dense_15: [(10, 64), (64,)]
  dense_16: [(64, 3), (3,)]

(Save/load commands commented out to avoid file creation)


## 11. GPU / Device Management

TensorFlow automatically uses GPU when available. Less manual device management than PyTorch.

In [25]:
# Check available devices
print("Physical devices:", tf.config.list_physical_devices())
print("GPUs:            ", tf.config.list_physical_devices('GPU'))

# TF automatically places operations on GPU if available
# No .to(device) needed! (Unlike PyTorch)

# Manual device placement (rarely needed):
with tf.device('/CPU:0'):
    a = tf.random.normal((100, 100))
    
# with tf.device('/GPU:0'):
#     b = tf.random.normal((100, 100))

print(f"\nTensor device: {a.device}")

# Memory growth — prevent TF from grabbing all GPU memory at once
# gpus = tf.config.list_physical_devices('GPU')
# for gpu in gpus:
#     tf.config.experimental.set_memory_growth(gpu, True)

# KEY DIFFERENCE from PyTorch:
# PyTorch:     model.to(device), data.to(device) — explicit, every time
# TensorFlow:  automatic GPU placement — you rarely think about it
# This makes TF simpler for beginners but less explicit

Physical devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
GPUs:             []

Tensor device: /job:localhost/replica:0/task:0/device:CPU:0


## 12. Common Architectures — CNN, RNN, and Transformer Patterns

In [26]:
# CNN — Convolutional Neural Network (for images)
# Note: TF uses channels-LAST by default: (batch, H, W, C)
# PyTorch uses channels-FIRST: (batch, C, H, W)

cnn_model = keras.Sequential([
    # Convolutional layers
    layers.Conv2D(16, kernel_size=3, activation='relu', padding='same',
                  input_shape=(28, 28, 1)),       # MNIST: 28x28, 1 channel
    layers.MaxPooling2D(pool_size=2),              # 28→14
    layers.Conv2D(32, kernel_size=3, activation='relu', padding='same'),
    layers.MaxPooling2D(pool_size=2),              # 14→7
    
    # Fully connected layers
    layers.Flatten(),                               # (7, 7, 32) → (1568,)
    layers.Dropout(0.25),
    layers.Dense(128, activation='relu'),
    layers.Dense(10, activation='softmax')          # 10 digit classes
])

cnn_model.summary()

# Test with MNIST-like input
dummy_images = tf.random.normal((4, 28, 28, 1))
print(f"\nCNN output: {cnn_model(dummy_images).shape}")

C:\Users\Windows11\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 16)     │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 1568)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 1568)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 128)            │       200,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 206,922 (808.29 KB)

 Trainable params: 206,922 (808.29 KB)

 Non-trainable params: 0 (0.00 B)


CNN output: (4, 10)


In [27]:
# RNN / LSTM — for sequential data (text, time series)

lstm_model = keras.Sequential([
    layers.Embedding(input_dim=5000, output_dim=128),     # Vocab → vectors
    layers.LSTM(64, return_sequences=True),                # Returns all timesteps
    layers.LSTM(32),                                        # Returns only last timestep
    layers.Dense(2, activation='softmax')                   # Binary sentiment
])

lstm_model.summary()

# Test with text-like input: batch of 8, sequence length 50
dummy_text = tf.random.uniform((8, 50), 0, 5000, dtype=tf.int32)
print(f"\nLSTM output: {lstm_model(dummy_text).shape}")

# Key parameters:
# return_sequences=True  → output at every timestep (for stacking LSTMs)
# return_sequences=False → output only at last timestep (default, for classification)

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


LSTM output: (8, 2)


In [28]:
# Transformer — using Keras MultiHeadAttention

class SimpleTransformer(keras.Model):
    def __init__(self, vocab_size, d_model, num_heads, num_classes):
        super().__init__()
        self.embedding = layers.Embedding(vocab_size, d_model)
        self.pos_embedding = layers.Embedding(512, d_model)
        self.attention = layers.MultiHeadAttention(num_heads=num_heads, key_dim=d_model)
        self.norm = layers.LayerNormalization()
        self.ffn = keras.Sequential([
            layers.Dense(d_model * 2, activation='relu'),
            layers.Dense(d_model)
        ])
        self.pool = layers.GlobalAveragePooling1D()
        self.classifier = layers.Dense(num_classes, activation='softmax')
    
    def call(self, inputs, training=False):
        seq_len = tf.shape(inputs)[1]
        positions = tf.range(seq_len)
        x = self.embedding(inputs) + self.pos_embedding(positions)
        
        # Self-attention + residual connection
        attn_output = self.attention(x, x)
        x = self.norm(x + attn_output)
        x = x + self.ffn(x)       # FFN + residual
        
        x = self.pool(x)           # Average over sequence
        return self.classifier(x)

transformer = SimpleTransformer(vocab_size=5000, d_model=64, num_heads=4, num_classes=2)
dummy_input = tf.random.uniform((8, 30), 0, 5000, dtype=tf.int32)
print("Transformer output:", transformer(dummy_input).shape)

Transformer output: (8, 2)


## 13. Real Patterns You'll See in AI Code

In [29]:
# Pattern 1: Transfer learning with a pretrained model
# THE most common pattern in modern TF projects

# Using a pretrained MobileNetV2 (commented — would download weights)
# base_model = keras.applications.MobileNetV2(
#     weights='imagenet',
#     include_top=False,            # Remove the classification head
#     input_shape=(224, 224, 3)
# )
# base_model.trainable = False     # Freeze pretrained weights
#
# model = keras.Sequential([
#     base_model,
#     layers.GlobalAveragePooling2D(),
#     layers.Dense(128, activation='relu'),
#     layers.Dropout(0.3),
#     layers.Dense(num_classes, activation='softmax')
# ])

# Demonstrating the freeze/unfreeze pattern:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dense(32, activation='relu'),
    layers.Dense(3, activation='softmax')
])
model(tf.random.normal((1, 20)))  # Build

# Freeze all layers
for layer in model.layers:
    layer.trainable = False

# Unfreeze only the last layer
model.layers[-1].trainable = True

trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
total = sum(tf.size(w).numpy() for w in model.weights)
print(f"Trainable: {trainable} / {total} parameters ({trainable/total:.1%})")

Trainable: 99 / 3523 parameters (2.8%)


In [30]:
# Pattern 2: Custom training loop with GradientTape
# Used in research code when model.fit() isn't flexible enough

model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dense(3)  # No softmax — use from_logits=True in loss
])

optimizer = keras.optimizers.Adam(learning_rate=0.001)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)

# Custom training step
@tf.function  # Compiles to a computation graph for speed
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = loss_fn(y, logits)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss

# Custom training loop
X_train = np.random.randn(200, 20).astype(np.float32)
y_train = np.random.randint(0, 3, 200)

dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).batch(32)

for epoch in range(3):
    total_loss = 0
    for step, (x_batch, y_batch) in enumerate(dataset):
        loss = train_step(x_batch, y_batch)
        total_loss += loss
    print(f"Epoch {epoch+1}: avg loss = {total_loss / (step+1):.4f}")

Epoch 1: avg loss = 1.2056
Epoch 2: avg loss = 1.1396
Epoch 3: avg loss = 1.0939


In [31]:
# Pattern 3: Multi-input / multi-output model (Functional API)
# Common in recommendation systems, multi-task learning

# Two inputs: user features and item features
user_input = keras.Input(shape=(10,), name='user_features')
item_input = keras.Input(shape=(5,), name='item_features')

# Process each input separately
user_branch = layers.Dense(32, activation='relu')(user_input)
item_branch = layers.Dense(32, activation='relu')(item_input)

# Merge
merged = layers.Concatenate()([user_branch, item_branch])
x = layers.Dense(64, activation='relu')(merged)

# Two outputs: rating prediction + click probability
rating_output = layers.Dense(1, name='rating')(x)
click_output = layers.Dense(1, activation='sigmoid', name='click')(x)

multi_model = keras.Model(
    inputs=[user_input, item_input],
    outputs=[rating_output, click_output]
)

# Compile with different losses for each output
multi_model.compile(
    optimizer='adam',
    loss={'rating': 'mse', 'click': 'binary_crossentropy'},
    loss_weights={'rating': 1.0, 'click': 0.5}  # Weight the losses
)

print("Multi-input/output model:")
multi_model.summary()

Multi-input/output model:


Model: "functional_9"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_features       │ (None, 10)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_features       │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_28 (Dense)    │ (None, 32)        │        352 │ user_features[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_29 (Dense)    │ (None, 32)        │        192 │ item_features[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ dense_28[0][0],   │
│ (Concatenate)       │                   │            │ dense_29[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_30 (Dense)    │ (None, 64)        │      4,160 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rating (Dense)      │ (None, 1)         │         65 │ dense_30[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ click (Dense)       │ (None, 1)         │         65 │ dense_30[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 4,834 (18.88 KB)

 Trainable params: 4,834 (18.88 KB)

 Non-trainable params: 0 (0.00 B)

In [32]:
# Pattern 4: Custom layer
# When built-in layers don't do what you need

class ScaledDotProductAttention(keras.layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
    
    def call(self, query, key, value):
        d_k = tf.cast(tf.shape(key)[-1], tf.float32)
        scores = tf.matmul(query, key, transpose_b=True) / tf.sqrt(d_k)
        weights = tf.nn.softmax(scores, axis=-1)
        return tf.matmul(weights, value)

# Test
attn = ScaledDotProductAttention()
q = tf.random.normal((2, 5, 8))   # (batch, seq_len, d_model)
k = tf.random.normal((2, 5, 8))
v = tf.random.normal((2, 5, 8))
print("Custom attention output:", attn(q, k, v).shape)

Custom attention output: (2, 5, 8)


In [33]:
# Pattern 5: Custom loss and metrics

class FocalLoss(keras.losses.Loss):
    """Focal Loss — focuses on hard examples. Used in object detection."""
    def __init__(self, gamma=2.0, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
    
    def call(self, y_true, y_pred):
        ce = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
        pt = tf.exp(-ce)
        return tf.reduce_mean((1 - pt) ** self.gamma * ce)

# Usage — drop-in replacement
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(20,)),
    layers.Dense(3, activation='softmax')
])
model.compile(optimizer='adam', loss=FocalLoss(gamma=2.0), metrics=['accuracy'])

X = np.random.randn(100, 20).astype(np.float32)
y = np.random.randint(0, 3, 100)
model.fit(X, y, epochs=3, verbose=0)
print("Custom Focal Loss training complete")

Custom Focal Loss training complete


In [34]:
# Pattern 6: @tf.function — graph compilation for speed
# You'll see this decorator on performance-critical functions

@tf.function
def compute_loss(model, x, y):
    predictions = model(x, training=False)
    return keras.losses.sparse_categorical_crossentropy(y, predictions)

# First call: traces and compiles to a graph (slow)
# Subsequent calls: runs the compiled graph (fast)
x = tf.random.normal((32, 20))
y = tf.constant(np.random.randint(0, 3, 32))
loss = compute_loss(model, x, y)
print(f"Loss: {tf.reduce_mean(loss).numpy():.4f}")

# Key rules for @tf.function:
# - Avoid Python side effects (print, append to list) — use tf.print instead
# - Avoid creating tf.Variables inside — create them outside
# - Inputs with different shapes will cause retracing (slow)
# - Great for training steps and inference functions

Loss: 1.2531


In [35]:
# Pattern 7: Reproducibility
tf.random.set_seed(42)
np.random.seed(42)

print("Seed set. Results will be reproducible.")
print("Random after seed:", tf.random.normal((3,)).numpy())

tf.random.set_seed(42)
print("Same seed again: ", tf.random.normal((3,)).numpy())

Seed set. Results will be reproducible.
Random after seed: [ 0.3274685 -0.8426258  0.3194337]
Same seed again:  [ 0.3274685 -0.8426258  0.3194337]


In [36]:
# Pattern 8: Mixed precision training
# Uses float16 for speed, float32 for stability — major speedup on GPU

# keras.mixed_precision.set_global_policy('mixed_float16')
# 
# model = keras.Sequential([
#     layers.Dense(64, activation='relu', input_shape=(20,)),
#     layers.Dense(3)   # Output stays float32 for loss stability
# ])
# # Output layer should produce float32 — add a cast layer or use from_logits
# model.compile(optimizer='adam',
#               loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True))
#
# # Reset to default after experiments:
# keras.mixed_precision.set_global_policy('float32')

print("Mixed precision: keras.mixed_precision.set_global_policy('mixed_float16')")
print("Requires GPU for actual speedup — shown as pseudocode")

Mixed precision: keras.mixed_precision.set_global_policy('mixed_float16')
Requires GPU for actual speedup — shown as pseudocode


## Quick Reference: PyTorch vs TensorFlow

| Concept | PyTorch | TensorFlow/Keras | Notes |
|---|---|---|---|
| Import | `import torch` | `import tensorflow as tf` | |
| Create tensor | `torch.tensor([1,2])` | `tf.constant([1,2])` | TF tensors are immutable |
| Mutable tensor | `torch.tensor(...)` | `tf.Variable(...)` | For model weights |
| Random normal | `torch.randn(3,4)` | `tf.random.normal((3,4))` | Tuple vs args |
| Type cast | `t.float()` | `tf.cast(t, tf.float32)` | |
| Reshape | `t.view(3,4)` | `tf.reshape(t, (3,4))` | |
| Add dim | `t.unsqueeze(0)` | `tf.expand_dims(t, 0)` | |
| Remove dim | `t.squeeze()` | `tf.squeeze(t)` | |
| Concatenate | `torch.cat` | `tf.concat` | |
| Clip | `torch.clamp` | `tf.clip_by_value` | |
| Sum | `t.sum(dim=0)` | `tf.reduce_sum(t, axis=0)` | `dim=` vs `axis=` |
| Max index | `t.argmax(dim=1)` | `tf.argmax(t, axis=1)` | |
| To NumPy | `t.detach().cpu().numpy()` | `t.numpy()` | TF is simpler |
| Boolean mask | `a[a > 5]` | `tf.boolean_mask(a, a > 5)` | |

## PyTorch vs TensorFlow: Training Patterns

| Concept | PyTorch | TensorFlow/Keras |
|---|---|---|
| **Define model** | `class Model(nn.Module)` | `keras.Sequential` / `keras.Model` |
| **Forward method** | `def forward(self, x)` | `def call(self, inputs, training)` |
| **Compile** | N/A (manual loop) | `model.compile(optimizer, loss)` |
| **Train** | Manual 5-step loop | `model.fit(X, y, epochs=10)` |
| **Evaluate** | Manual with `torch.no_grad()` | `model.evaluate(X, y)` |
| **Predict** | `model(x)` with `no_grad` | `model.predict(x)` |
| **Train/eval mode** | `model.train()` / `model.eval()` | `training=True/False` per layer |
| **Save model** | `torch.save(state_dict)` | `model.save('model.keras')` |
| **GPU** | `model.to(device)` — explicit | Automatic |
| **Gradients** | `loss.backward()` | `tape.gradient(loss, vars)` |
| **Early stopping** | Manual implementation | `callbacks.EarlyStopping` |
| **LR scheduling** | `optim.lr_scheduler.*` | `callbacks.ReduceLROnPlateau` |
| **Data loading** | `DataLoader(Dataset)` | `tf.data.Dataset` |
| **Graph mode** | `torch.jit.script` | `@tf.function` |
| **Channels** | NCHW (batch, C, H, W) | NHWC (batch, H, W, C) |